# Assignment #9: Model Deployment
## Preprocessing, Hyperparameter Tuning & Model Deployment: California Housing

Workflow: load data -> preprocess with `ColumnTransformer` -> 80/20 split -> KNN regressor -> `GridSearchCV` (5-fold, R²) -> evaluate -> save pipeline.

## 0. Imports

In [1]:
import numpy as np
import pandas as pd
import pickle
import joblib

from sklearn.datasets import fetch_california_housing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

## 1. Load the dataset using `fetch_california_housing`

In [2]:
housing = fetch_california_housing(as_frame=True)
X = housing.data          # 8 features (DataFrame)
y = housing.target        # MedHouseVal, in $100,000s

print("Shape of X:", X.shape)
print("Features:", list(X.columns))
X.head()

Shape of X: (20640, 8)
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
# Quick look at the data
print(X.describe().T)
print("\nMissing values per column:\n", X.isnull().sum())

              count         mean          std         min         25%  \
MedInc      20640.0     3.870671     1.899822    0.499900    2.563400   
HouseAge    20640.0    28.639486    12.585558    1.000000   18.000000   
AveRooms    20640.0     5.429000     2.474173    0.846154    4.440716   
AveBedrms   20640.0     1.096675     0.473911    0.333333    1.006079   
Population  20640.0  1425.476744  1132.462122    3.000000  787.000000   
AveOccup    20640.0     3.070655    10.386050    0.692308    2.429741   
Latitude    20640.0    35.631861     2.135952   32.540000   33.930000   
Longitude   20640.0  -119.569704     2.003532 -124.350000 -121.800000   

                    50%          75%           max  
MedInc         3.534800     4.743250     15.000100  
HouseAge      29.000000    37.000000     52.000000  
AveRooms       5.229129     6.052381    141.909091  
AveBedrms      1.048780     1.099526     34.066667  
Population  1166.000000  1725.000000  35682.000000  
AveOccup       2.818116 

## 2. Split the data into training and test sets (80/20)

The split is done **before** any fitting of the preprocessor, so no information from the test set leaks into the scaler.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (16512, 8)  Test: (4128, 8)


## 3. Why scaling matters for KNN: comparing scalers

KNN is distance-based, so features on large scales (like `Population`) dominate features on small scales (like `AveBedrms`). Here we compare no scaling vs. `StandardScaler`, `MinMaxScaler` and `RobustScaler` using a baseline `n_neighbors=5`.

In [5]:
# -----------------------------------
# No Scaling
# -----------------------------------
model_raw = KNeighborsRegressor(n_neighbors=5)
model_raw.fit(X_train, y_train)
y_pred_raw = model_raw.predict(X_test)
r2_raw = r2_score(y_test, y_pred_raw)
print(f"Without Scaling: R² = {r2_raw:.4f}")

# -----------------------------------
# StandardScaler
# -----------------------------------
scaler_standard = StandardScaler()
X_train_std = scaler_standard.fit_transform(X_train)
X_test_std = scaler_standard.transform(X_test)

# -----------------------------------
# MinMaxScaler
# -----------------------------------
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

# -----------------------------------
# RobustScaler
# -----------------------------------
scaler_robust = RobustScaler()
X_train_robust = scaler_robust.fit_transform(X_train)
X_test_robust = scaler_robust.transform(X_test)

# Evaluate each scaled version with the same baseline model
results = {"No scaling": r2_raw}
for name, (Xtr, Xte) in {
    "StandardScaler": (X_train_std, X_test_std),
    "MinMaxScaler": (X_train_minmax, X_test_minmax),
    "RobustScaler": (X_train_robust, X_test_robust),
}.items():
    m = KNeighborsRegressor(n_neighbors=5)
    m.fit(Xtr, y_train)
    results[name] = r2_score(y_test, m.predict(Xte))

for name, score in results.items():
    print(f"{name:<15} R² = {score:.4f}")

Without Scaling: R² = 0.1463
No scaling      R² = 0.1463
StandardScaler  R² = 0.6700
MinMaxScaler    R² = 0.6895
RobustScaler    R² = 0.6758


## 4. Preprocessing with `ColumnTransformer`

All 8 features in this dataset are numeric (there are no categorical columns), so the `ColumnTransformer` applies `StandardScaler` to the numeric features. The structure is the same one you would extend with a `OneHotEncoder` branch for categorical data.

In [6]:
numeric_features = list(X.columns)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features)
    # A categorical branch would go here, e.g.:
    # ('cat', OneHotEncoder(drop='first'), categorical_features)
])

# Pipeline = preprocessing + model. Putting the scaler inside the pipeline means
# GridSearchCV re-fits it on each CV training fold (no leakage across folds).
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor())
])
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['MedInc', 'HouseAge',
                                                   'AveRooms', 'AveBedrms',
                                                   'Population', 'AveOccup',
                                                   'Latitude',
                                                   'Longitude'])])),
                ('knn', KNeighborsRegressor())])

## 5. `GridSearchCV` with 5-fold cross-validation (scoring = R²)

Grid: `n_neighbors` in [3, 5, 7, 9], `weights` in ['uniform', 'distance'], `p` in [1, 2]. That is 4 x 2 x 2 = 16 combinations x 5 folds = 80 fits. Parameter names use the `knn__` prefix because the model is a step inside the pipeline.

In [7]:
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['MedInc',
                                                                          'HouseAge',
                                                                          'AveRooms',
                                                                          'AveBedrms',
                                                                          'Population',
                                                                          'AveOccup',
                                                                          'Latitude',
                                                                          'Longitude'])])),
                                       ('knn', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'knn__n_neighbors': [3, 5, 7, 9], 'knn__p': [1, 2],
                         'knn__weights': ['uniform', 'distance']},
             scoring='r2', verbose=1)

## 6. Best parameters and test-set evaluation

In [8]:
print("Best Parameters:", grid.best_params_)
print(f"Best CV R² Score: {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
test_r2 = r2_score(y_test, y_pred)
print(f"Test Set R² Score: {test_r2:.4f}")

Best Parameters: {'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV R² Score: 0.7313
Test Set R² Score: 0.7221


In [9]:
# All 16 combinations ranked by mean CV R²
cv_results = pd.DataFrame(grid.cv_results_)
cv_results[['param_knn__n_neighbors', 'param_knn__weights', 'param_knn__p',
            'mean_test_score', 'std_test_score', 'rank_test_score']] \
    .sort_values('rank_test_score').head(10)

,param_knn__n_neighbors,param_knn__weights,param_knn__p,mean_test_score,std_test_score,rank_test_score
13,9,distance,1,0.731267,0.003097,1
9,7,distance,1,0.729590,0.002959,2
12,9,uniform,1,0.726708,0.002832,3
8,7,uniform,1,0.725730,0.002272,4
5,5,distance,1,0.724431,0.005392,5
4,5,uniform,1,0.721714,0.004978,6
1,3,distance,1,0.697983,0.006403,7
15,9,distance,2,0.697230,0.005236,8
0,3,uniform,1,0.695842,0.005695,9
11,7,distance,2,0.695755,0.006120,10


## 7. Save the trained pipeline (pickle and joblib)

In [10]:
# Save with pickle
with open('best_knn_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)
    print("Model saved as 'best_knn_model.pkl'")

# Also save with joblib
joblib.dump(best_model, 'best_knn_model.joblib')
print("Model saved as 'best_knn_model.joblib'")

# Load back and confirm it reproduces the same predictions
with open('best_knn_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

y_loaded_pred = loaded_model.predict(X_test)
print("R² score using loaded model:", r2_score(y_test, y_loaded_pred))

Model saved as 'best_knn_model.pkl'
Model saved as 'best_knn_model.joblib'
R² score using loaded model: 0.72210916268423


In [11]:
# Try a single prediction with the loaded pipeline (raw, unscaled input;
# the pipeline scales it internally)
sample = pd.DataFrame([{
    'MedInc': 5.0, 'HouseAge': 20.0, 'AveRooms': 6.0, 'AveBedrms': 1.0,
    'Population': 1500.0, 'AveOccup': 3.0, 'Latitude': 34.0, 'Longitude': -118.0
}])
pred = loaded_model.predict(sample)[0]
print(f"Predicted median house value: {pred:.3f}  (= ${pred*100000:,.0f})")

Predicted median house value: 2.152  (= $215,232)


## 8. Download the saved model

Run this cell to download the saved pipeline to your computer (also available from the Files panel on the left).

In [12]:
from google.colab import files
import sklearn
files.download('best_knn_model.pkl')
print("scikit-learn version used:", sklearn.__version__)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

scikit-learn version used: 1.6.1


## 9. Summary

- **Preprocessing:** `ColumnTransformer` with `StandardScaler` inside a `Pipeline`, so scaling is fit only on training folds.
- **Model:** `KNeighborsRegressor`, tuned with `GridSearchCV` (5-fold CV, R² scoring) over `n_neighbors`, `weights`, `p`.
- **Evaluation:** best parameters and CV R² printed above, final R² reported on the held-out 20% test set.
- **Deployment:** pipeline saved as `best_knn_model.pkl` / `.joblib`, ready to be loaded for prediction.